# DEXTRA · 02 — Build the modelling dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/02_build_dataset.ipynb)

Turns the raw downloads from notebook 01 into the table the models train on, and saves it back to Hugging Face.

| Section | Builds | Needs |
|---|---|---|
| A | Daily meteorology per city: boundary layer, wind, inversion index | `raw/era5_surface_grid/`, `raw/era5_pressure/` |
| B | Daily fire counts within 100 km and 500 km of each city | `raw/firms/` |
| C | Daily city pollutants and CPCB AQI | **`raw/cpcb/`** |
| D | Merge, calendar and city features, splits, scaling | A + B + C |
| E | Checks: coverage, class balance, physical sanity | D |

**Sections A and B run today.** Section C needs the CPCB station files, which cannot be scripted — the CPCB repository is behind a CAPTCHA, so those files are downloaded in a browser and uploaded to `raw/cpcb/`. Until then this notebook builds and saves the meteorology and fire tables, then stops with a note. Nothing is wasted: re-run it once CPCB is there.

### Runtime
No GPU needed. Use a **CPU** runtime and keep your free T4 hours for training.

### Secrets
The same two as notebook 01: `HF_TOKEN` and `HF_DATASET_REPO` (🔑 icon in the left sidebar, **Notebook access** on).


## 0 · Settings

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

RUN_METEO = True        # Section A
RUN_FIRE = True         # Section B
RUN_POLLUTANTS = True   # Section C (needs raw/cpcb/)
RUN_ASSEMBLE = True     # Section D
REBUILD = False         # True re-makes outputs that already exist

## 1 · Secrets, packages and code

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from src import config
from src.data import assemble, daily_cpcb, era5_features, fire_features
from src.data.cpcb_loader import load_station_file, parse_filename
from src.utils.hf_store import HFStore

store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
YEARS = list(range(int(config.DATA_START_DATE[:4]), int(config.DATA_END_DATE[:4]) + 1))
START, END = config.DATA_START_DATE, config.DATA_END_DATE
CITY_SLUG = {city: city.lower() for city in config.TARGET_CITIES}

def have(path):
    return store.exists(path) and not REBUILD

print(f"Connected to {HF_DATASET_REPO}: {len(store.files())} files")
print(f"Window {START} to {END}, {len(config.TARGET_CITIES)} cities")

## A · Daily meteorology
Hourly ERA5 becomes one row per city per day, grouped by **Indian** calendar days so a day means the same span CPCB uses.

Two features carry the physics the Mixture-of-Experts gate routes on:
- **`inv_idx`** — temperature at the city's own inversion level minus 2 m temperature, in the early morning. Positive means warm air caps cool air and pollution cannot rise. Each city uses the level about 75 hPa above its own surface, because a fixed 925 hPa is underground in Bengaluru.
- **`blh_anomaly`** — boundary-layer height against that city's own 30-day normal, so stagnation is judged locally.

`blh_min`, the night-time minimum, is kept alongside the daily mean — see section E for why that matters.

In [ ]:
METEO_PATH = "processed/meteo_daily.parquet"

if RUN_METEO and not have(METEO_PATH):
    surface_by_city = [store.read_df(f"raw/era5_surface_grid/{CITY_SLUG[c]}.parquet")
                       for c in config.TARGET_CITIES]
    pressure_files = sorted(store.files("raw/era5_pressure/"))
    pressure = pd.concat([store.read_df(p) for p in pressure_files], ignore_index=True)
    print(f"pressure-level years loaded: {[p.rsplit('/', 1)[1][:4] for p in pressure_files]}")

    meteo = era5_features.build_meteo_daily(surface_by_city, pressure)
    store.upload_df(meteo, METEO_PATH, "Daily meteorology per city")
    print(f"↑ {METEO_PATH}: {len(meteo):,} city-days")
    display(meteo.groupby("city")[["inversion_level_hpa"]].first().T)
elif RUN_METEO:
    print(f"✓ {METEO_PATH} already built")

## B · Daily fire counts
Delhi's worst air arrives from crop fires 200–400 km upwind in Punjab and Haryana, so each city counts detections in two rings: **100 km** for local burning and **500 km** for regional transport.

Only confident *vegetation* fires count. FIRMS also flags volcanoes and static land sources such as gas flares, which burn every day and would read as a permanently elevated fire count for industrial cities rather than a burning season.

In [ ]:
FIRE_PATH = "processed/fire_daily.parquet"

if RUN_FIRE and not have(FIRE_PATH):
    fire_files = sorted(store.files("raw/firms/"))
    frames = []
    for path in fire_files:
        frames.append(fire_features.city_fire_daily(store.read_df(path)))
        print(f"  {path.rsplit('/', 1)[1]} done")
    fire_daily = fire_features.fill_calendar(pd.concat(frames, ignore_index=True), START, END)
    store.upload_df(fire_daily, FIRE_PATH, "Daily fire counts per city")
    print(f"↑ {FIRE_PATH}: {len(fire_daily):,} city-days")

    peak = (fire_daily.assign(month=fire_daily.date.dt.month)
                      .pivot_table(index="month", columns="city", values="fire_count_500km", aggfunc="sum"))
    shown = [c for c in ["Delhi", "Lucknow", "Chennai", "Guwahati"] if c in peak.columns] or list(peak.columns)[:4]
    display(peak[shown].astype(int))
elif RUN_FIRE:
    print(f"✓ {FIRE_PATH} already built")

## C · Daily city pollutants and AQI
Each station's hourly file becomes daily values by CPCB's own rules: PM2.5, PM10, NO2, SO2 and NH3 take the 24-hour mean, while **CO and O₃ take the highest 8-hour running mean**, which is what CPCB's breakpoints for those two are defined on. A day needs 16 valid hours, so a mostly-dead sensor produces no number instead of a misleading one.

Stations are then combined by **median**, and `n_stations` records how many actually reported. That column matters: Guwahati and Bhopal run on a single station for years, so their "city median" is one sensor, and the paper has to say so.

Files are expected at `raw/cpcb/<city>/Raw_Data_<year>_site_<id>_<station>_1Hr.csv`.

In [ ]:
POLLUTANT_PATH = "processed/pollutant_daily.parquet"
cpcb_files = store.files("raw/cpcb/")

if RUN_POLLUTANTS and not cpcb_files:
    print("✗ No CPCB files found under raw/cpcb/.\n"
          "  The CPCB repository is behind a CAPTCHA, so these are downloaded in a browser\n"
          "  and uploaded separately. Sections A and B above are saved; re-run this notebook\n"
          "  once raw/cpcb/ exists.")
elif RUN_POLLUTANTS and not have(POLLUTANT_PATH):
    import collections, tempfile
    from huggingface_hub import hf_hub_download

    by_city = collections.defaultdict(list)
    for path in cpcb_files:
        if path.endswith(".csv"):
            by_city[path.split("/")[2]].append(path)

    city_frames, skipped = [], []
    for slug, paths in sorted(by_city.items()):
        city = next((c for c, s in CITY_SLUG.items() if s == slug), slug)
        dailies = []
        for path in sorted(paths):
            local = hf_hub_download(HF_DATASET_REPO, path, repo_type="dataset", token=HF_TOKEN)
            hourly = load_station_file(local)
            daily = daily_cpcb.station_daily(hourly)
            if daily[[c for c in config.POLLUTANTS if c in daily.columns]].notna().any().any():
                dailies.append(daily)
            else:
                skipped.append(path.rsplit("/", 1)[1])
        if dailies:
            city_frames.append(daily_cpcb.city_daily(dailies, city))
        print(f"  {city:<11} {len(dailies)} station-years with data, {len(paths) - len(dailies)} empty")

    pollutants = daily_cpcb.add_aqi(pd.concat(city_frames, ignore_index=True))
    pollutants = daily_cpcb.reindex_to_calendar(pollutants, START, END)
    store.upload_df(pollutants, POLLUTANT_PATH, "Daily city pollutants and CPCB AQI")
    print(f"\n↑ {POLLUTANT_PATH}: {len(pollutants):,} city-days, "
          f"AQI on {pollutants.AQI.notna().mean():.1%} of them, {len(skipped)} empty station-years skipped")
elif RUN_POLLUTANTS:
    print(f"✓ {POLLUTANT_PATH} already built")

## D · Assemble, split and scale
Merges the three daily tables, adds calendar and city features, assigns splits and fits the scalers.

Two things here decide whether the headline result is believable:
- **Scalers are fitted on the training split only**, and saved as `processed/scaler_params.json` — the Android app needs exactly these numbers.
- **Static city features respect the few-shot boundary.** For the five unseen cities they are computed from the first 30 days of that city's data alone; using seven years would quietly undo the "adapts with 30 days" claim.

In [ ]:
DATASET_PATH = "processed/city_daily_features.parquet"

needed = ["processed/pollutant_daily.parquet", METEO_PATH, FIRE_PATH]
if RUN_ASSEMBLE and not all(store.exists(p) for p in needed):
    print("✗ Waiting on:", [p for p in needed if not store.exists(p)])
elif RUN_ASSEMBLE:
    daily = assemble.merge_sources(store.read_df(needed[0]), store.read_df(METEO_PATH), store.read_df(FIRE_PATH))
    daily = assemble.add_calendar_features(daily)

    static = assemble.city_static_features(daily)
    daily = assemble.add_city_meta(daily, static)
    daily["split"] = assemble.assign_split(daily)

    params = assemble.fit_scalers(daily[daily.split == "train"])
    scaled = assemble.apply_scalers(daily, params)

    store.upload_df(daily, DATASET_PATH, "City-daily features, unscaled")
    store.upload_df(scaled, "processed/city_daily_scaled.parquet", "City-daily features, scaled")
    store.upload_json(params, "processed/scaler_params.json", "Scaling parameters fitted on train")
    store.upload_df(static, "meta/city_static.csv", "Static per-city features")

    episodes = assemble.extreme_episodes(daily)
    store.upload_df(episodes, "processed/extreme_episodes.parquet", "Hazardous AQI episodes")

    print(f"↑ {DATASET_PATH}: {len(daily):,} city-days x {len(assemble.feature_columns())} model variates")
    print(f"↑ extreme_episodes: {len(episodes)} episodes above AQI {config.HAZARDOUS_AQI_THRESHOLD:.0f}")
    display(daily.groupby("split").agg(rows=("date", "size"), cities=("city", "nunique"),
                                       first=("date", "min"), last=("date", "max")))

## E · Checks
### E1 · Coverage and the rare-event balance
The share of hazardous days is the number the whole project rests on: if extremes were common, there would be nothing to augment.

In [ ]:
if store.exists(DATASET_PATH):
    daily = store.read_df(DATASET_PATH)
    rows = []
    for city, g in daily.groupby("city"):
        rated = g.AQI.notna()
        rows.append({
            "city": city,
            "split": "unseen" if city in config.TEST_UNSEEN_CITIES else "train",
            "days with AQI": int(rated.sum()),
            "coverage %": round(100 * rated.mean(), 1),
            "median stations": int(g.n_stations.median()),
            f"> {config.HAZARDOUS_AQI_THRESHOLD:.0f} %": round(100 * (g.AQI > config.HAZARDOUS_AQI_THRESHOLD).sum() / max(rated.sum(), 1), 1),
            f"> {config.RARE_AQI_THRESHOLD:.0f} %": round(100 * (g.AQI > config.RARE_AQI_THRESHOLD).sum() / max(rated.sum(), 1), 1),
        })
    summary = pd.DataFrame(rows).sort_values(f"> {config.RARE_AQI_THRESHOLD:.0f} %", ascending=False)
    display(summary)

    rare = 100 * (daily.AQI > config.RARE_AQI_THRESHOLD).sum() / daily.AQI.notna().sum()
    print(f"Overall share of days above AQI {config.RARE_AQI_THRESHOLD:.0f}: {rare:.1f}%")
    thin = summary[summary[f"> {config.RARE_AQI_THRESHOLD:.0f} %"] < 1.0]["city"].tolist()
    if thin:
        print(f"⚠ Almost no rare events in: {thin}. Rare-event recall cannot be measured there — "
              "worth deciding before these are used as test cities.")
else:
    print("Nothing to check yet: section D has not produced the dataset.")

### E2 · Does the physics hold?
Correlations between the meteorological features and AQI. Signs matter more than sizes: an inversion should trap pollution, wind should disperse it.

In [ ]:
if store.exists(DATASET_PATH):
    valid = daily.dropna(subset=["AQI"])
    rows = []
    for column, expected in [("inv_idx", "+ traps"), ("blh", "- disperses"), ("blh_min", "- disperses"),
                             ("blh_anomaly", "- disperses"), ("wind_speed", "- ventilates"),
                             ("precip", "- washes out"), ("fire_count_500km", "+ adds smoke")]:
        if column in valid.columns:
            r = valid[column].corr(valid.AQI)
            rows.append({"feature": column, "corr with AQI": round(r, 3), "expected": expected,
                         "agrees": "✓" if (r > 0) == expected.startswith("+") else "✗"})
    display(pd.DataFrame(rows))
    print("A '✗' is not automatically wrong: full-year correlations mix seasons, and within winter\n"
          "some of these reverse. That seasonal conditioning is the argument for regime-switching experts.")
else:
    print("Nothing to check yet: section D has not produced the dataset.")

### E3 · The worst episodes
The events DEXTRA has to predict. These become the case study in the paper.

In [ ]:
if store.exists("processed/extreme_episodes.parquet"):
    episodes = store.read_df("processed/extreme_episodes.parquet")
    print(f"{len(episodes)} episodes, longest {episodes.duration_days.max()} days, "
          f"highest peak {episodes.peak_aqi.max():.0f}")
    display(episodes.nlargest(10, "peak_aqi"))
    display(episodes.groupby(episodes.start_date.dt.month).size().rename("episodes starting in month").to_frame().T)
else:
    print("Nothing to check yet: section D has not produced the dataset.")

## Next
1. **`03_eda.ipynb`** — the figures and numbers the paper rests on, including how many hazardous days each test city actually has.
2. **`04_baselines.ipynb`** — persistence, XGBoost, LSTM and a plain iTransformer. That is the decision point: if simple models already handle extreme days well, the central claim needs rethinking, and it is far better to learn that now than in week 8.